<a href="https://colab.research.google.com/github/deacoki/ai-mastery-2026/blob/main/Integration_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import requests
import pandas as pd
from datetime import datetime

HISTORY_FILE = "daily_reports_history.csv"

# ==================== Safe API Helper ====================
def safe_get(url, timeout=10):
    try:
        response = requests.get(url, timeout=timeout)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"⚠️ Error fetching {url}: {e}")
        return None

# ==================== NEW: Get Location from User ====================
def get_user_location():
    print("🌍 Choose your location (US-friendly defaults)\n")

    city = input("Enter city name (default: Seattle): ").strip()
    if not city:
        city = "Seattle"

    # Default coordinates for common US cities
    defaults = {
        "Seattle": (47.6062, -122.3321, "America/Los_Angeles"),
        "New York": (40.7128, -74.0060, "America/New_York"),
        "Los Angeles": (34.0522, -118.2437, "America/Los_Angeles"),
        "Chicago": (41.8781, -87.6298, "America/Chicago"),
        "Austin": (30.2672, -97.7431, "America/Chicago"),
        "Miami": (25.7617, -80.1918, "America/New_York"),
        "Missoula": (46.8721, -113.9940, "America/Denver")
    }

    if city in defaults:
        latitude, longitude, timezone = defaults[city]
        print(f"✅ Using saved coordinates for {city}")
    else:
        print(f"📍 Enter coordinates for {city}:")
        try:
            lat_input = input("Latitude (default 47.6062): ").strip()
            latitude = float(lat_input) if lat_input else 47.6062

            lon_input = input("Longitude (default -122.3321): ").strip()
            longitude = float(lon_input) if lon_input else -122.3321
        except ValueError:
            print("⚠️ Invalid number. Using Seattle defaults.")
            latitude, longitude = 47.6062, -122.3321

        timezone = input("Timezone (default America/Los_Angeles): ").strip() or "America/Los_Angeles"

    print(f"📍 Using: {city} (Lat: {latitude}, Lon: {longitude})")
    return city, latitude, longitude, timezone

# ==================== Temperature Conversion ====================
def c_to_f(celsius):
    """Convert Celsius to Fahrenheit"""
    if celsius is None:
        return None
    return round((celsius * 9/5) + 32, 1)

# ==================== Get Weather (Now in Fahrenheit + Motivation) ====================
def get_weather(city, latitude, longitude, timezone):
    url = f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current_weather=true&daily=temperature_2m_max,temperature_2m_min&timezone={timezone}"

    data = safe_get(url)
    if not data:
        return {
            "summary": f"🌤️ Weather unavailable for {city}",
            "temp_f": None,
            "motivation": "Stay motivated anyway! 💪"
        }

    try:
        current_c = data["current_weather"]["temperature"]
        max_c = data["daily"]["temperature_2m_max"][0]
        min_c = data["daily"]["temperature_2m_min"][0]

        # Convert to Fahrenheit
        current_f = c_to_f(current_c)
        max_f = c_to_f(max_c)
        min_f = c_to_f(min_c)

        # Motivational Level based on Fahrenheit (US-friendly)
        if current_f < 40:
            motivation = "❄️ Bundle up! It's cold out there — perfect for hot coffee and focus!"
        elif current_f < 55:
            motivation = "🧥 Chilly day — great weather for getting stuff done indoors!"
        elif current_f < 70:
            motivation = "😊 Pleasant and mild — ideal day to crush your goals!"
        elif current_f < 85:
            motivation = "☀️ Warm and energizing — go make big progress today!"
        else:
            motivation = "🔥 It's hot! Stay hydrated and keep that momentum going!"

        summary = f"🌡️ {city} today: {current_f}°F (High {max_f}°F / Low {min_f}°F)"

        return {
            "summary": summary,
            "temp_f": current_f,
            "motivation": motivation
        }
    except Exception as e:
        print(f"⚠️ Error processing weather: {e}")
        return {
            "summary": f"🌤️ Weather unavailable for {city}",
            "temp_f": None,
            "motivation": "Stay motivated anyway! 💪"
        }

# ==================== Get Quote ====================
def get_quote():
    data = safe_get("https://zenquotes.io/api/random")
    if data and isinstance(data, list) and len(data) > 0:
        q = data[0]
        return f"💡 \"{q['q']}\" — {q['a']}"
    return "💡 No quote available today."

# ==================== Personal Note with Validation ====================
def get_personal_note():
    print("\n📝 Personal Note Section")
    while True:
        note = input("Enter a quick personal note for today (or press Enter to skip): ").strip()
        if len(note) > 200:
            print("⚠️ Note is too long (max 200 characters). Please shorten it.")
        else:
            return note if note else "No personal note today."

# ==================== Load & Append to History ====================
def load_and_update_history(new_row):
    try:
        if pd.io.common.file_exists(HISTORY_FILE):
            history = pd.read_csv(HISTORY_FILE)
            history = pd.concat([history, new_row], ignore_index=True)
            history = history.drop_duplicates(subset=["Date"], keep="last")
            print(f"📊 History updated — Total reports: {len(history)}")
        else:
            history = new_row
            print("📂 Created new history file.")

        history.to_csv(HISTORY_FILE, index=False)
        return history
    except Exception as e:
        print(f"⚠️ Error updating history: {e}")
        return new_row

# ==================== Main Daily Report ====================
print("🌟 === My Daily AI Report (US Edition) === 🌟\n")

# Get user location
city, latitude, longitude, timezone = get_user_location()

print(f"📅 Date: {datetime.now().strftime('%Y-%m-%d')}\n")

weather = get_weather(city, latitude, longitude, timezone)
quote = get_quote()
note = get_personal_note()

# Nice formatted output
print("\n" + "="*70)
print(weather["summary"])
print(weather["motivation"])
print("\n" + quote)
print(f"\n📝 Personal Note: {note}")
print("="*70)

# ==================== Create Today's Report Row ====================
report_data = {
    "Date": [datetime.now().date()],
    "City": [city],
    "Weather": [weather["summary"]],
    "Motivational_Level": [weather["motivation"]],
    "Quote": [quote],
    "Personal_Note": [note]
}

today_df = pd.DataFrame(report_data)

# Update history
history_df = load_and_update_history(today_df)

# ==================== Save Today's Report ====================
filename_base = f"daily_report_{datetime.now().strftime('%Y%m%d')}"

today_df.to_csv(f"{filename_base}.csv", index=False)

with open(f"{filename_base}.txt", "w") as f:
    f.write(f"🌟 DAILY AI REPORT - {datetime.now().strftime('%Y-%m-%d')} 🌟\n")
    f.write(f"📍 Location: {city}\n")
    f.write("="*70 + "\n\n")
    f.write(weather["summary"] + "\n")
    f.write(weather["motivation"] + "\n\n")
    f.write(quote + "\n\n")
    f.write(f"📝 Personal Note:\n{note}\n")
    f.write("="*70 + "\n")

print(f"\n✅ Today's report saved as:")
print(f"   • {filename_base}.csv")
print(f"   • {filename_base}.txt")
print(f"✅ History saved/updated in {HISTORY_FILE}")

🌟 === My Daily AI Report (US Edition) === 🌟

🌍 Choose your location (US-friendly defaults)

Enter city name (default: Seattle): Missoula
✅ Using saved coordinates for Missoula
📍 Using: Missoula (Lat: 46.8721, Lon: -113.994)
📅 Date: 2026-04-09


📝 Personal Note Section
Enter a quick personal note for today (or press Enter to skip): Love Thursday!

🌡️ Missoula today: 50.9°F (High 61.2°F / Low 34.0°F)
🧥 Chilly day — great weather for getting stuff done indoors!

💡 "Nothing is impossible. The word itself says 'I'm possible!'" — Audrey Hepburn

📝 Personal Note: Love Thursday!
📊 History updated — Total reports: 2

✅ Today's report saved as:
   • daily_report_20260409.csv
   • daily_report_20260409.txt
✅ History saved/updated in daily_reports_history.csv
